# Dauphin Island spring migrants: build the filtered, zero-filled dataset

This notebook takes the statewide eBird table (`ebd`, compact download for US-AL) and produces:

- **`checklists`**: one row per checklist that passes the quality filters, with effort covariates
- **`zerofilled`**: one row per checklist × target species, with `detected` (0/1) and `count`

Your original database is opened **read-only** and is never changed. Results go to a new DuckDB file and to Parquet.

Run the cells top to bottom. Edit only the **Settings** cell.

In [ ]:
# Setup: pip install -r requirements.txt  (see README)
import os
import duckdb
from dotenv import load_dotenv

load_dotenv()   # reads machine-specific paths from .env (git-ignored); see .env.example

## 1. Settings

- **`SRC_DB`**: path to your existing `.duckdb` file with the `ebd` table.
- **`SED_FILE`**: optional path to the `..._sampling.txt` file from the download. With it, checklists that recorded no birds are included. Leave it as `None` to build the checklist list from `ebd`, which misses only those empty checklists.
- **Bounding box**: approximate. Check it on a map and tighten it.

In [ ]:
SRC_DB   = os.getenv("EBD_SRC_DB", "ebd_US-AL_smp_relAug-2026_compact_opt.duckdb")   # set in .env
TABLE    = "ebd"
SED_FILE = os.getenv("EBD_SED_FILE") or None   # optional sampling-event file, set in .env

OUT_DB   = os.getenv("DAUPHIN_DB", "dauphin.duckdb")
OUT_DIR  = os.getenv("DATA_DIR", "data")

# Dauphin Island bounding box (approximate)
LAT_MIN, LAT_MAX = 30.20, 30.29
LON_MIN, LON_MAX = -88.36, -88.06

YEAR_MIN = 2008

# Spring window, inclusive, as (month, day)
SPRING_START = (3, 15)
SPRING_END   = (5, 20)

# Effort filters (eBird best-practice defaults)
MAX_DURATION_MIN = 300    # 5 hours
MAX_DISTANCE_KM  = 5
MAX_OBSERVERS    = 10

# Target species: names exactly as they appear in COMMON NAME
TARGET_SPECIES = [
    "Indigo Bunting",
    "Rose-breasted Grosbeak",
    "Scarlet Tanager",
    "Summer Tanager",
    "Wood Thrush",
    "Hooded Warbler",
    "Kentucky Warbler",
    "Ruby-throated Hummingbird",
    # resident "control" species
    "Northern Cardinal",
    "Carolina Wren",
]

# Grid size for spatial subsampling, in degrees (~0.01 deg = ~1 km)
GRID_DEG = 0.01

## 2. Connect

This creates (or opens) the output database and attaches your original one read-only as `src`.

> If you get a *"could not set lock"* error, close any other program that has one of these files open, such as another notebook kernel, DBeaver or the DuckDB CLI.

In [ ]:
os.makedirs(OUT_DIR, exist_ok=True)
q = lambda p: p.replace("'", "''")   # escape quotes in file paths for SQL

con = duckdb.connect(OUT_DB)
con.execute("DETACH DATABASE IF EXISTS src")
con.execute(f"ATTACH '{q(SRC_DB)}' AS src (READ_ONLY)")
SRC = f'src.main."{TABLE}"'

con.sql(f"SELECT COUNT(*) AS sightings, COUNT(DISTINCT \"SAMPLING EVENT IDENTIFIER\") AS checklists FROM {SRC}").show()

## 3. Filter checklists

The filters, all applied per checklist:

| Filter | Why |
|---|---|
| `ALL SPECIES REPORTED` is true | Only complete checklists can be zero-filled |
| Protocol `P21` (Stationary) or `P22` (Traveling) | Drops incidental and other non-standard protocols |
| Inside the Dauphin Island box | Study area |
| Year ≥ `YEAR_MIN`, inside the spring window | Study period |
| Duration ≤ 5 h, distance ≤ 5 km, observers ≤ 10 | Removes extreme-effort checklists |
| Start time and duration recorded | Needed as model covariates |

Column names contain spaces, so they're wrapped in double quotes. Shared group checklists (same `GROUP IDENTIFIER`) are reduced to one copy.

In [ ]:
(sm, sd), (em, ed) = SPRING_START, SPRING_END

checklist_filter = f'''
        "ALL SPECIES REPORTED"
    AND "PROTOCOL CODE" IN ('P21', 'P22')
    AND "LATITUDE"  BETWEEN {LAT_MIN} AND {LAT_MAX}
    AND "LONGITUDE" BETWEEN {LON_MIN} AND {LON_MAX}
    AND year("OBSERVATION DATE") >= {YEAR_MIN}
    AND (   (month("OBSERVATION DATE") = {sm} AND day("OBSERVATION DATE") >= {sd})
         OR (month("OBSERVATION DATE") > {sm} AND month("OBSERVATION DATE") < {em})
         OR (month("OBSERVATION DATE") = {em} AND day("OBSERVATION DATE") <= {ed}))
    AND "DURATION MINUTES" IS NOT NULL AND "DURATION MINUTES" <= {MAX_DURATION_MIN}
    AND COALESCE("EFFORT DISTANCE KM", 0) <= {MAX_DISTANCE_KM}
    AND "NUMBER OBSERVERS" IS NOT NULL AND "NUMBER OBSERVERS" <= {MAX_OBSERVERS}
    AND "TIME OBSERVATIONS STARTED" IS NOT NULL
'''

checklist_cols = '''
    "SAMPLING EVENT IDENTIFIER"         AS checklist_id,
    "GROUP IDENTIFIER"                  AS group_identifier,
    "OBSERVER ID"                       AS observer_id,
    "LOCALITY ID"                       AS locality_id,
    "LOCALITY"                          AS locality,
    "LATITUDE"                          AS latitude,
    "LONGITUDE"                         AS longitude,
    "OBSERVATION DATE"                  AS obs_date,
    "TIME OBSERVATIONS STARTED"         AS start_time,
    "PROTOCOL CODE"                     AS protocol,
    "DURATION MINUTES"                  AS duration_min,
    COALESCE("EFFORT DISTANCE KM", 0)   AS distance_km,   -- stationary counts = 0 km
    "NUMBER OBSERVERS"                  AS n_observers
'''

if SED_FILE:
    con.execute(f'''
        CREATE OR REPLACE TEMP TABLE sed AS
        SELECT * FROM read_csv('{q(SED_FILE)}', delim = '\t', header = true, quote = '', sample_size = -1)
    ''')
    base = f"SELECT DISTINCT {checklist_cols} FROM sed WHERE {checklist_filter}"
else:
    # The compact EBD repeats checklist info on every sighting row; DISTINCT collapses it.
    base = f"SELECT DISTINCT {checklist_cols} FROM {SRC} WHERE {checklist_filter}"

con.execute(f'''
    CREATE OR REPLACE TABLE checklists AS
    WITH f AS ({base})
    SELECT *,
        year(obs_date)                                AS year,
        dayofyear(obs_date)                           AS day_of_year,
        hour(start_time) + minute(start_time) / 60.0  AS start_hour,
        weekofyear(obs_date)                          AS week,
        floor(latitude  / {GRID_DEG})::INTEGER        AS cell_y,
        floor(longitude / {GRID_DEG})::INTEGER        AS cell_x
    FROM f
    QUALIFY row_number() OVER (
        PARTITION BY COALESCE(group_identifier, checklist_id)
        ORDER BY checklist_id) = 1
''')

con.sql("SELECT COUNT(*) AS checklists_kept FROM checklists").show()

## 4. Sightings on those checklists

The compact download has no `CATEGORY` column, so non-species entries are dropped by name: spuhs (`warbler sp.`), slashes (`Tree/Barn Swallow`), hybrids (`... x ...`) and domestic types.

Each checklist also gets **`n_species`**, the number of species it recorded, which serves as a proxy for observer skill and effort.

In [ ]:
con.execute(f'''
    CREATE OR REPLACE TABLE obs AS
    SELECT "SAMPLING EVENT IDENTIFIER" AS checklist_id,
           "COMMON NAME"               AS common_name,
           "OBSERVATION COUNT"         AS observation_count
    FROM {SRC}
    WHERE "APPROVED"
      AND "SAMPLING EVENT IDENTIFIER" IN (SELECT checklist_id FROM checklists)
      AND "COMMON NAME" NOT LIKE '% sp.%'
      AND "COMMON NAME" NOT LIKE '%/%'
      AND "COMMON NAME" NOT LIKE '% x %'
      AND "COMMON NAME" NOT LIKE '%Domestic%'
''')

con.execute('''
    CREATE OR REPLACE TABLE checklists AS
    SELECT c.*, COALESCE(r.n_species, 0) AS n_species
    FROM checklists c
    LEFT JOIN (SELECT checklist_id, COUNT(DISTINCT common_name) AS n_species
               FROM obs GROUP BY checklist_id) r USING (checklist_id)
''')

con.sql("SELECT COUNT(*) AS sightings_kept, COUNT(DISTINCT common_name) AS species FROM obs").show()

## 5. Zero-fill

Every checklist is crossed with every target species. Where the species wasn't reported, `detected = 0`.

`count` is `NULL` when the observer recorded **X** (present, not counted).

**Check the warning:** any species "never detected" is almost always a spelling mismatch with eBird's names.

In [ ]:
con.execute("CREATE OR REPLACE TABLE target_species (common_name VARCHAR)")
con.executemany("INSERT INTO target_species VALUES (?)", [[s] for s in TARGET_SPECIES])

missing = [r[0] for r in con.execute('''
    SELECT common_name FROM target_species
    WHERE common_name NOT IN (SELECT DISTINCT common_name FROM obs)
''').fetchall()]
if missing:
    print("WARNING - never detected (check spelling):", missing)

con.execute('''
    CREATE OR REPLACE TABLE zerofilled AS
    WITH o AS (   -- collapse subspecies rows to one row per checklist x species
        SELECT checklist_id, common_name,
               CASE WHEN bool_or(observation_count = 'X') THEN NULL
                    ELSE SUM(TRY_CAST(observation_count AS INTEGER)) END AS count
        FROM obs
        GROUP BY checklist_id, common_name
    )
    SELECT c.*,
           s.common_name,
           CASE WHEN o.checklist_id IS NULL THEN 0 ELSE 1 END AS detected,
           CASE WHEN o.checklist_id IS NULL THEN 0 ELSE o.count END AS count
    FROM checklists c
    CROSS JOIN target_species s
    LEFT JOIN o ON o.checklist_id = c.checklist_id AND o.common_name = s.common_name
''')

con.sql("SELECT COUNT(*) AS rows, COUNT(DISTINCT checklist_id) AS checklists FROM zerofilled").show()

## 6. Sanity checks

Look for years with very few checklists (consider raising `YEAR_MIN`) and for target species with very low reporting rates (consider dropping them).

In [ ]:
checklists_per_year = con.sql('''
    SELECT year, COUNT(*) AS n_checklists
    FROM checklists GROUP BY year ORDER BY year
''').df()
checklists_per_year

In [ ]:
con.sql('''
    SELECT common_name,
           SUM(detected)               AS detections,
           ROUND(AVG(detected), 3)     AS reporting_rate
    FROM zerofilled
    GROUP BY common_name
    ORDER BY reporting_rate DESC
''').df()

In [ ]:
# Quick look at effort growth over time (needs matplotlib)
ax = checklists_per_year.plot(x="year", y="n_checklists", kind="bar", legend=False,
                              title="Filtered spring checklists per year, Dauphin Island")
ax.set_ylabel("checklists");

## 7. Save to Parquet

Teammates can query these files directly, e.g. `duckdb.sql("SELECT * FROM 'data/zerofilled.parquet'")`, without opening the database.

In [ ]:
for t in ("checklists", "zerofilled"):
    path = os.path.join(OUT_DIR, f"{t}.parquet")
    con.execute(f"COPY {t} TO '{q(path)}' (FORMAT parquet)")
    print("saved", path)

## 8. Spatial subsampling (for later analysis)

Keeps one checklist per ~1 km grid cell per week per year, which reduces the pull of heavily birded hotspots. Use a different `seed` for each bootstrap or robustness run.

In [ ]:
def spatial_subsample(seed=1):
    return con.sql(f'''
        SELECT * FROM checklists
        QUALIFY row_number() OVER (
            PARTITION BY year, week, cell_x, cell_y
            ORDER BY hash(checklist_id || '{seed}')) = 1
    ''').df()

sub = spatial_subsample(seed=1)
print(f"{len(sub)} of {con.execute('SELECT COUNT(*) FROM checklists').fetchall()[0][0]} checklists kept after subsampling")

## 9. Load the result for analysis

`zerofilled` as a pandas DataFrame, ready for the trend and covariate models.

In [ ]:
zf = con.sql("SELECT * FROM zerofilled").df()
zf.head()

In [ ]:
con.close()   # release the file lock when you're done